# Day 8 - NLP Ticket Classification

Classify tickets into 30 categories using short description only.

**Key Finding:** Only 44 distinct short descriptions in 48,503 tickets.

The ceiling is a property of the DATA, not the MODEL.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score

In [ ]:
# Load data
df = pd.read_csv('../data/day_6.csv')
df['opened_at'] = pd.to_datetime(df['opened_at'])
df = df.drop_duplicates(subset=['short_description', 'description', 'category'])

print(f'Shape: {df.shape}')
print(f'Classes: {df.category.nunique()}')
print(f'Distinct short descriptions: {df.short_description.nunique()}')

In [ ]:
# Temporal split
tr = df[df.opened_at < '2026-05-01']
va = df[(df.opened_at >= '2026-05-01') & (df.opened_at < '2026-07-01')]
te = df[df.opened_at >= '2026-07-01']

X_tr, y_tr = tr.short_description, tr.category
X_va, y_va = va.short_description, va.category
X_te, y_te = te.short_description, te.category

print(f'Train: {len(tr)}, Val: {len(va)}, Test: {len(te)}')

In [ ]:
# TF-IDF + LinearSVC
svm = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)),
    ('svc', LinearSVC(C=1.0, random_state=0, max_iter=5000))
])

svm.fit(X_tr, y_tr)
pred_va = svm.predict(X_va)

acc = accuracy_score(y_va, pred_va)
f1 = f1_score(y_va, pred_va, average='macro', zero_division=0)

print(f'Validation - Accuracy: {acc:.4f}, Macro-F1: {f1:.4f}')

In [ ]:
# Test set evaluation
pred_te = svm.predict(X_te)
acc_te = accuracy_score(y_te, pred_te)
f1_te = f1_score(y_te, pred_te, average='macro', zero_division=0)

print(f'Test - Accuracy: {acc_te:.4f}, Macro-F1: {f1_te:.4f}')

## Summary

- Only 44 distinct short descriptions across 48,503 rows
- Ceiling accuracy: 0.6756 (majority vote lookup)
- Both TF-IDF+SVM and embeddings achieve this ceiling
- Model choice doesn't matter when input is poor quality
- **Better input > Better model**
- Adding description field increases Macro-F1 from 0.26 to 0.57